# Where the Money Goes

Financial analysis: calculating revenue, fixed and variable costs, contribution margins, operating profit and profit margin.
Profitability of each product, the biggest expenses, the break-even point, how sensitive the profit is to costs, charts and financial conclusions.



In [1]:
from pathlib import Path
from google.colab import files

REQUIRED_FILES = ['clean_sales.csv', 'product_profitability.csv', 'monthly_profit.csv', 'operating_costs.csv']
missing = [name for name in REQUIRED_FILES if not Path('/content', name).is_file()]
if missing:
    print('Select these CSV files:', ', '.join(missing))
    uploaded = files.upload()
else:
    print('All required CSV files are already uploaded.')

still_missing = [name for name in REQUIRED_FILES if not Path('/content', name).is_file()]
if still_missing:
    raise FileNotFoundError('Missing CSV files: ' + ', '.join(still_missing))
print('Ready! All required CSV files are present.')

All required CSV files are already uploaded.
Ready! All required CSV files are present.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/content') if Path('/content').is_dir() else Path.cwd()
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
TABLES = PROJECT_ROOT / 'reports' / 'tables'
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

def read_data(folder, filename):
    path = PROJECT_ROOT / filename
    if not path.is_file():
        raise FileNotFoundError(f'Missing {filename}. Run the upload cell at the top of this notebook.')
    df = pd.read_csv(path)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
    return df

def save_plot(filename):
    plt.tight_layout()
    plt.savefig(FIGURES / filename, dpi=150, bbox_inches='tight')
    plt.close()

def save_table(df, filename):
    df.to_csv(TABLES / filename, index=False)

def check_close(actual, expected, label, tolerance=0.06):
    if not np.isclose(actual, expected, atol=tolerance):
        raise ValueError(f'{label}: {actual:.2f} != {expected:.2f}')

def quality_report(datasets):
    for name, df in datasets.items():
        print(f'{name}: {len(df)} rows; missing={df.isna().sum().sum()}; duplicates={df.duplicated().sum()}')

pd.set_option('display.float_format', '{:,.2f}'.format)
print('Working folder:', PROJECT_ROOT)


def break_even(monthly, total_items, n_days):
    revenue = monthly['revenue'].sum()
    variable = monthly['variable_cost'].sum()
    fees = monthly['royalty'].sum() + monthly['marketing_fee'].sum()
    fixed = monthly[['rent','salaries','utilities','other_costs']].sum().sum()
    net_ratio = (revenue - variable - fees) / revenue
    if net_ratio <= 0:
        raise ValueError('Non-positive net contribution ratio')
    required_revenue = fixed / net_ratio
    avg_revenue_per_item = revenue / total_items
    return {
        'net_contribution_ratio': net_ratio,
        'fixed_costs': fixed,
        'break_even_revenue_period': required_revenue,
        'break_even_revenue_month': required_revenue / len(monthly),
        'margin_of_safety_pct': (revenue - required_revenue) / revenue * 100,
        'break_even_items_per_day': required_revenue / avg_revenue_per_item / n_days,
        'actual_items_per_day': total_items / n_days,
    }


Working folder: /content


## 1. Load financial data

In [3]:
sales = read_data("processed", "clean_sales.csv")
products = read_data("processed", "product_profitability.csv")
monthly = read_data("processed", "monthly_profit.csv")
fixed_costs = read_data("simulated", "operating_costs.csv")

monthly["month"] = monthly["month"].astype(str)
fixed_costs["month"] = fixed_costs["month"].astype(str)
monthly

,month,revenue,variable_cost,contribution_margin,rent,salaries,utilities,other_costs,royalty,marketing_fee,operating_profit,operating_margin_pct
0,2026-01,48240,"16,751.51","31,488.49",5000,12000,1900,1500,"1,929.60",723.60,"8,435.29",17.49
1,2026-02,49137,"17,096.04","32,040.96",5000,12000,1800,1500,"1,965.48",737.06,"9,038.42",18.39
2,2026-03,53566,"18,594.38","34,971.62",5000,12000,1600,1500,"2,142.64",803.49,"11,925.49",22.26


## 2. Validate the numbers

In [4]:
# revenue, variable cost and contribution margin must be the same in all three files
for column in ["revenue", "variable_cost", "contribution_margin"]:
    check_close(sales[column].sum(), monthly[column].sum(), f"sales vs monthly: {column}")
    check_close(sales[column].sum(), products[column].sum(), f"sales vs products: {column}")

In [5]:
# the fixed costs in the monthly table must be the same as the input file
for column in ["rent", "salaries", "utilities", "other_costs"]:
    merged = monthly[["month", column]].merge(
        fixed_costs[["month", column]], on="month", suffixes=("_statement", "_input"), validate="one_to_one"
    )
    assert np.allclose(merged[column + "_statement"], merged[column + "_input"])
print("Fixed costs match.")

# recompute the operating profit myself and compare
monthly["recomputed_profit"] = monthly["contribution_margin"] - monthly[
    ["rent", "salaries", "utilities", "other_costs", "royalty", "marketing_fee"]
].sum(axis=1)
assert np.allclose(monthly["recomputed_profit"], monthly["operating_profit"], atol=0.06)
print("Operating profit recomputed correctly.")

Fixed costs match.
Operating profit recomputed correctly.


## 3. Revenue and profit

In [6]:
quarter_revenue = monthly["revenue"].sum()
quarter_variable_cost = monthly["variable_cost"].sum()
quarter_contribution = monthly["contribution_margin"].sum()
quarter_profit = monthly["operating_profit"].sum()

print(f"Total revenue: {quarter_revenue:,.2f} RON")
print(f"Variable costs: {quarter_variable_cost:,.2f} RON")
print(f"Contribution margin: {quarter_contribution:,.2f} RON ({quarter_contribution / quarter_revenue * 100:.1f}% of revenue)")
print(f"Operating profit: {quarter_profit:,.2f} RON ({quarter_profit / quarter_revenue * 100:.1f}% of revenue)")

Total revenue: 150,943.00 RON
Variable costs: 52,441.93 RON
Contribution margin: 98,501.07 RON (65.3% of revenue)
Operating profit: 29,399.20 RON (19.5% of revenue)


## 4. Monthly profit and loss

In [7]:
pl_columns = ["month", "revenue", "variable_cost", "contribution_margin", "royalty", "marketing_fee",
              "rent", "salaries", "utilities", "other_costs", "operating_profit", "operating_margin_pct"]
monthly[pl_columns]

,month,revenue,variable_cost,contribution_margin,royalty,marketing_fee,rent,salaries,utilities,other_costs,operating_profit,operating_margin_pct
0,2026-01,48240,"16,751.51","31,488.49","1,929.60",723.60,5000,12000,1900,1500,"8,435.29",17.49
1,2026-02,49137,"17,096.04","32,040.96","1,965.48",737.06,5000,12000,1800,1500,"9,038.42",18.39
2,2026-03,53566,"18,594.38","34,971.62","2,142.64",803.49,5000,12000,1600,1500,"11,925.49",22.26


In [8]:
monthly.plot.bar(x="month", y=["revenue", "operating_profit"], figsize=(10, 5),
                 title="Monthly Revenue and Operating Profit", rot=0, color=["tan", "saddlebrown"])
plt.xlabel("Month")
plt.ylabel("RON")
save_plot("finance_monthly_results.png")

## 5. Cost structure

In [9]:
cost_columns = ["variable_cost", "royalty", "marketing_fee", "rent", "salaries", "utilities", "other_costs"]

cost_mix = (
    monthly[cost_columns].sum()
    .sort_values(ascending=False)
    .rename_axis("cost_type")
    .reset_index(name="ron")
)
cost_mix["share_of_revenue_pct"] = cost_mix["ron"] / quarter_revenue * 100
cost_mix

,cost_type,ron,share_of_revenue_pct
0,variable_cost,"52,441.93",34.74
1,salaries,"36,000.00",23.85
2,rent,"15,000.00",9.94
3,royalty,"6,037.72",4.00
4,utilities,"5,300.00",3.51
5,other_costs,"4,500.00",2.98
6,marketing_fee,"2,264.15",1.50


In [10]:
cost_mix.plot.bar(x="cost_type", y="ron", figsize=(11, 5), legend=False,
                  title="Operating Cost Structure", color="peru", rot=20)
plt.xlabel("Cost Category")
plt.ylabel("Cost (RON)")
save_plot("finance_cost_structure.png")

## 6. Product contribution

In [11]:
products = products.sort_values("contribution_margin", ascending=False)
products["margin_per_item"] = products["contribution_margin"] / products["quantity"]
products["share_of_contribution_pct"] = products["contribution_margin"] / products["contribution_margin"].sum() * 100
products

,product_id,product_name,category,quantity,revenue,variable_cost,contribution_margin,margin_pct,margin_per_item,share_of_contribution_pct
2,3,Cappuccino,Coffee,6898,55184,"20,694.00","34,490.00",62.50,5.00,35.01
3,4,Latte,Coffee,5166,46494,"17,461.08","29,032.92",62.44,5.62,29.47
0,1,Espresso,Coffee,6702,33510,"9,717.90","23,792.10",71.00,3.55,24.15
1,2,Americano,Coffee,3151,15755,"4,568.95","11,186.05",71.00,3.55,11.36


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
products.plot.bar(x="product_name", y="contribution_margin", ax=axes[0], legend=False, color="saddlebrown", rot=0)
axes[0].set_title("Contribution Margin by Product (RON)")
axes[0].set_xlabel("")
products.plot.bar(x="product_name", y="margin_pct", ax=axes[1], legend=False, color="tan", rot=0)
axes[1].set_title("Contribution Margin (% of price)")
axes[1].set_xlabel("")
save_plot("finance_product_contribution.png")

## 7. Break-even

In [13]:
daily = read_data("processed", "daily_sales.csv")
total_items = sales["quantity"].sum()
n_days = sales["date"].nunique()

be = break_even(monthly, total_items, n_days)

print(f"Left from each 1 RON of revenue after raw materials, royalty and marketing fee: {be['net_contribution_ratio']:.3f} RON")
print(f"Fixed costs (3 months): {be['fixed_costs']:,.2f} RON")
print(f"Break-even revenue, 3 months: {be['break_even_revenue_period']:,.2f} RON")
print(f"Break-even revenue, per month: {be['break_even_revenue_month']:,.2f} RON")
print(f"Margin of safety: {be['margin_of_safety_pct']:.1f}%")
print(f"Break-even items per day: {be['break_even_items_per_day']:.0f}  (actual: {be['actual_items_per_day']:.0f})")

# how many days were under the break-even level?
daily["under_break_even"] = daily["quantity"] < be["break_even_items_per_day"]
print()
print("Share of days under break-even (0 = weekday, 1 = weekend):")
print(daily.groupby("is_weekend")["under_break_even"].mean().round(2))

Left from each 1 RON of revenue after raw materials, royalty and marketing fee: 0.598 RON
Fixed costs (3 months): 60,800.00 RON
Break-even revenue, 3 months: 101,745.19 RON
Break-even revenue, per month: 33,915.06 RON
Margin of safety: 32.6%
Break-even items per day: 164  (actual: 244)

Share of days under break-even (0 = weekday, 1 = weekend):
is_weekend
0   0.00
1   0.08
Name: under_break_even, dtype: float64


*Note: this break-even uses the average over the period. A single quiet day under the line is not a loss for the month, because the
fixed costs are paid monthly.*

## 8. What if raw materials get more expensive?

In [14]:
rows = []
for increase in [0, 0.05, 0.10, 0.15, 0.20]:
    profit = quarter_profit - quarter_variable_cost * increase
    rows.append({
        "variable_cost_increase_pct": increase * 100,
        "estimated_quarter_profit": profit,
        "profit_change_pct": (profit / quarter_profit - 1) * 100,
    })
cost_sensitivity = pd.DataFrame(rows)
cost_sensitivity

,variable_cost_increase_pct,estimated_quarter_profit,profit_change_pct
0,0.00,"29,399.20",0.00
1,5.00,"26,777.10",-8.92
2,10.00,"24,155.01",-17.84
3,15.00,"21,532.91",-26.76
4,20.00,"18,910.81",-35.68


In [15]:
cost_sensitivity.plot(x="variable_cost_increase_pct", y="estimated_quarter_profit", figsize=(9, 5),
                      title="Impact of Higher Variable Costs on Profit", marker="o", color="saddlebrown", legend=False)
plt.xlabel("Variable Cost Increase (%)")
plt.ylabel("Estimated Operating Profit (RON)")
save_plot("finance_cost_sensitivity.png")

## 9. Export the tables

In [16]:
save_table(monthly, "finance_monthly_reconciled.csv")
save_table(cost_mix, "finance_cost_mix.csv")
save_table(products, "finance_product_contribution.csv")
save_table(cost_sensitivity, "finance_cost_sensitivity.csv")

## What I found
In 3 months the café earns about 29,400 RON, which is 19.5% of revenue. Raw materials (about 35 RON out of 100) and salaries
(about 24 RON out of 100) are the biggest costs, and salaries do not change with sales, so busy days matter a lot.
Cappuccino and latte give the most RON per cup and about 64% of the contribution. The café breaks even at about 164 items a day and sells about 244.
The biggest danger is the price of raw materials: +10% takes away about 18% of the profit.